# Fine-tuning ruBERT для разметки релевантности Telegram-постов

Ноутбук подготовлен для Kaggle. Он обучает BERT-модель для бинарной классификации `relevant`: пост релевантен хотя бы одному из пяти посылов ЦБ РФ или не релевантен.

Используются готовые файлы:

- `bert_relevance_corpus_train.parquet` - train-часть корпуса;
- `bert_relevance_corpus_test.parquet` - финальная test-часть корпуса.

В старом варианте ноутбук решал multi-label задачу по `t1..t6`. Здесь задача другая: главный таргет один, `relevant`. Колонки `t1_relevant` ... `t5_relevant` оставлены для контроля распределений и анализа ошибок, но модель обучается именно на бинарный таргет.


## 1. Ключевые правила корпуса

Из документации `data/annotated_data/doc.md`:

- `text` - входной текст Telegram-поста для BERT.
- `relevant` - главный бинарный таргет: `1`, если пост релевантен хотя бы одной из пяти тем, иначе `0`.
- `annotated=True` - пост реально видел LLM-разметчик. Такие нули считаются надежными негативами.
- `annotated=False` - пост был отсечен префильтром и назначен негативом по построению. Среди таких нулей возможен шум.
- В корпусе сильный дисбаланс: позитивов мало, поэтому ниже используется взвешенная BCE-потеря и подбор порога по validation.

Рекомендуемый режим по умолчанию - `TRAINING_MODE = "clean"`: учим и валидируемся только на `annotated=True`. Это дает более чистые метки. Для эксперимента можно включить `"hybrid"`, где `annotated=False` добавляются только в train, но validation остается чистым.


## 2. Импорты, seed и базовые настройки


In [ ]:
from pathlib import Path
import inspect
import json
import os
import random
import re
import warnings

import numpy as np
import pandas as pd

SEED = 42


def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)

    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True
    except Exception:
        pass


seed_everything(SEED)
warnings.filterwarnings("default")

pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_columns", 120)

print("Seed:", SEED)
print("Working directory:", Path.cwd())


## 3. Поиск train/test файлов на Kaggle

Загрузите оба parquet-файла как Kaggle Dataset. Код ниже сам ищет файлы по именам внутри `/kaggle/input`. Если вы храните данные в другом месте, задайте `MANUAL_TRAIN_PATH` и `MANUAL_TEST_PATH` вручную.


In [ ]:
TRAIN_FILENAME = "bert_relevance_corpus_train.parquet"
TEST_FILENAME = "bert_relevance_corpus_test.parquet"

MANUAL_TRAIN_PATH = None
MANUAL_TEST_PATH = None

LOCAL_FALLBACKS = [
    Path("data/annotated_data"),
    Path("../data/annotated_data"),
]


def find_input_file(filename: str, manual_path: str | None = None) -> Path:
    if manual_path:
        path = Path(manual_path)
        if not path.exists():
            raise FileNotFoundError(f"MANUAL path does not exist: {path}")
        return path

    candidates: list[Path] = []
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        candidates.extend(sorted(kaggle_root.rglob(filename)))

    for root in LOCAL_FALLBACKS:
        path = root / filename
        if path.exists():
            candidates.append(path)

    if not candidates:
        raise FileNotFoundError(
            f"Не найден {filename}. Добавьте parquet-файлы как Kaggle Dataset "
            "или задайте MANUAL_TRAIN_PATH / MANUAL_TEST_PATH."
        )

    if len(candidates) > 1:
        print(f"Найдено несколько кандидатов для {filename}; выбран первый:")
        for path in candidates:
            print("  ", path)

    return candidates[0]


TRAIN_PATH = find_input_file(TRAIN_FILENAME, MANUAL_TRAIN_PATH)
TEST_PATH = find_input_file(TEST_FILENAME, MANUAL_TEST_PATH)

print("TRAIN_PATH:", TRAIN_PATH)
print("TEST_PATH:", TEST_PATH)


## 4. Загрузка и проверка схемы

Test-файл уже является отложенной выборкой. Мы не пересоздаем test, а только выделяем validation из train-файла.


In [ ]:
train_raw = pd.read_parquet(TRAIN_PATH)
test_raw = pd.read_parquet(TEST_PATH)

print("Raw train:", train_raw.shape)
print("Raw test :", test_raw.shape)
print("Train columns:")
print(train_raw.columns.tolist())

display(train_raw.head(3))


In [ ]:
TEXT_COL = "text"
TARGET_COL = "relevant"
ANNOTATED_COL = "annotated"
KEY_COLS = ["channel_id", "message_id"]
TOPIC_COLS = [f"t{i}_relevant" for i in range(1, 6)]
META_COLS = [
    "channel_id",
    "channel_name",
    "message_id",
    "post_date",
    "source",
    "annotated",
    "text_hash",
]

REQUIRED_COLS = [TEXT_COL, TARGET_COL, ANNOTATED_COL] + TOPIC_COLS


def normalize_telegram_text(value: object) -> str:
    text = "" if pd.isna(value) else str(value)
    text = text.replace("\u200b", " ").replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def validate_and_prepare(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    missing = [col for col in REQUIRED_COLS if col not in df.columns]
    if missing:
        raise ValueError(f"{split_name}: нет обязательных колонок: {missing}")

    out = df.copy()
    out[TEXT_COL] = out[TEXT_COL].map(normalize_telegram_text)
    out = out[out[TEXT_COL] != ""].copy()

    out[TARGET_COL] = out[TARGET_COL].astype("int8")
    bad_targets = sorted(set(out[TARGET_COL].dropna().unique()) - {0, 1})
    if bad_targets:
        raise ValueError(f"{split_name}: unexpected target values: {bad_targets}")

    out[ANNOTATED_COL] = out[ANNOTATED_COL].fillna(False).astype(bool)
    for col in TOPIC_COLS:
        out[col] = out[col].fillna(False).astype(bool)

    if KEY_COLS[0] in out.columns and KEY_COLS[1] in out.columns:
        duplicated = out.duplicated(KEY_COLS).sum()
        if duplicated:
            warnings.warn(f"{split_name}: найдено дублей по {KEY_COLS}: {duplicated}")

    noisy_positive = ((out[ANNOTATED_COL] == False) & (out[TARGET_COL] == 1)).sum()
    if noisy_positive:
        raise ValueError(f"{split_name}: annotated=False должен иметь relevant=0, найдено {noisy_positive}")

    out["text_len_chars"] = out[TEXT_COL].str.len().astype("int32")
    out["topic_combo"] = out[TOPIC_COLS].astype(int).astype(str).agg("".join, axis=1)
    return out.reset_index(drop=True)


train_raw = validate_and_prepare(train_raw, "train")
test_raw = validate_and_prepare(test_raw, "test")

if all(col in train_raw.columns for col in KEY_COLS) and all(col in test_raw.columns for col in KEY_COLS):
    train_keys = set(map(tuple, train_raw[KEY_COLS].to_numpy()))
    test_keys = set(map(tuple, test_raw[KEY_COLS].to_numpy()))
    overlap = train_keys & test_keys
    print("Train/test overlap by key:", len(overlap))
    if overlap:
        raise ValueError("Train и test пересекаются по (channel_id, message_id).")

print("Prepared train:", train_raw.shape)
print("Prepared test :", test_raw.shape)


## 5. Контроль распределений

Здесь проверяем баланс `relevant`, `annotated`, источников и пяти тем. Это помогает убедиться, что train/test подключены правильно и не перепутаны.


In [ ]:
def split_overview(df: pd.DataFrame, name: str) -> dict:
    positives = int(df[TARGET_COL].sum())
    annotated = int(df[ANNOTATED_COL].sum())
    return {
        "split": name,
        "rows": len(df),
        "positive": positives,
        "positive_share": positives / len(df),
        "negative": len(df) - positives,
        "annotated": annotated,
        "annotated_share": annotated / len(df),
        "median_chars": int(df["text_len_chars"].median()),
        "p95_chars": int(df["text_len_chars"].quantile(0.95)),
    }


overview = pd.DataFrame([
    split_overview(train_raw, "train_raw"),
    split_overview(test_raw, "test_raw"),
])
display(overview)

print("Source distribution, %:")
source_dist = pd.concat(
    [
        train_raw["source"].value_counts(normalize=True).rename("train_raw") if "source" in train_raw else pd.Series(dtype=float),
        test_raw["source"].value_counts(normalize=True).rename("test_raw") if "source" in test_raw else pd.Series(dtype=float),
    ],
    axis=1,
).fillna(0).sort_index() * 100
display(source_dist.round(2))

print("Topic positive counts:")
topic_counts = pd.DataFrame({
    "train_raw": train_raw[TOPIC_COLS].sum().astype(int),
    "test_raw": test_raw[TOPIC_COLS].sum().astype(int),
})
display(topic_counts)

print("Topic-combo top counts:")
display(pd.concat([
    train_raw["topic_combo"].value_counts().rename("train_raw"),
    test_raw["topic_combo"].value_counts().rename("test_raw"),
], axis=1).fillna(0).astype(int).head(15))


## 6. Формирование train / validation

`bert_relevance_corpus_test.parquet` остается финальным holdout.

Доступные режимы:

- `clean` - рекомендуемый: train и validation только из `annotated=True`.
- `hybrid` - validation из `annotated=True`, а в train добавляются также `annotated=False` негативы.
- `full` - train и validation из всех строк, включая шумные `annotated=False` негативы.

Для первой качественной модели начните с `clean`. Затем можно сравнить с `hybrid`.


In [ ]:
from sklearn.model_selection import train_test_split

TRAINING_MODE = "clean"  # "clean", "hybrid" или "full"
VALIDATION_SIZE = 0.15


def choose_stratify_key(df: pd.DataFrame) -> pd.Series | None:
    combo = df["topic_combo"].astype(str)
    combo_counts = combo.value_counts()
    if len(combo_counts) > 1 and combo_counts.min() >= 2:
        return combo

    if "source" in df.columns:
        source_target = df["source"].astype(str) + "_" + df[TARGET_COL].astype(str)
        source_target_counts = source_target.value_counts()
        if len(source_target_counts) > 1 and source_target_counts.min() >= 2:
            return source_target

    target = df[TARGET_COL].astype(str)
    target_counts = target.value_counts()
    if len(target_counts) > 1 and target_counts.min() >= 2:
        return target

    return None


def train_validation_split(raw_train: pd.DataFrame, mode: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    mode = mode.lower().strip()
    if mode not in {"clean", "hybrid", "full"}:
        raise ValueError("TRAINING_MODE должен быть one of: clean, hybrid, full")

    annotated_df = raw_train[raw_train[ANNOTATED_COL]].copy()
    unannotated_df = raw_train[~raw_train[ANNOTATED_COL]].copy()

    split_base = annotated_df if mode in {"clean", "hybrid"} else raw_train.copy()
    stratify_key = choose_stratify_key(split_base)
    train_part, val_df = train_test_split(
        split_base,
        test_size=VALIDATION_SIZE,
        random_state=SEED,
        shuffle=True,
        stratify=stratify_key,
    )

    if mode == "hybrid" and len(unannotated_df):
        train_df = pd.concat([train_part, unannotated_df], axis=0, ignore_index=True)
        train_df = train_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    else:
        train_df = train_part.reset_index(drop=True)

    val_df = val_df.reset_index(drop=True)
    return train_df, val_df


train_df, val_df = train_validation_split(train_raw, TRAINING_MODE)

display(pd.DataFrame([
    split_overview(train_df, "train_model"),
    split_overview(val_df, "validation"),
    split_overview(test_raw, "test_full"),
    split_overview(test_raw[test_raw[ANNOTATED_COL]].copy(), "test_clean"),
]))

print("TRAINING_MODE:", TRAINING_MODE)
print("Train positives:", int(train_df[TARGET_COL].sum()), "/", len(train_df))
print("Val positives:", int(val_df[TARGET_COL].sum()), "/", len(val_df))


## 7. Сохранение подготовленных split-файлов

Это не обязательно для обучения, но удобно для воспроизводимости: в `/kaggle/working` сохраняются именно те train/validation/test, которые использует ноутбук.


In [ ]:
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
PREPARED_DIR = WORK_DIR / "prepared_telegram_relevance_data"
PREPARED_DIR.mkdir(parents=True, exist_ok=True)

train_df.to_parquet(PREPARED_DIR / "train_model.parquet", index=False)
val_df.to_parquet(PREPARED_DIR / "validation.parquet", index=False)
test_raw.to_parquet(PREPARED_DIR / "test_full.parquet", index=False)
test_raw[test_raw[ANNOTATED_COL]].to_parquet(PREPARED_DIR / "test_clean.parquet", index=False)

print("Prepared data saved to:", PREPARED_DIR)


## 8. Модель и токенизатор

По умолчанию используется `DeepPavlov/rubert-base-cased`. Если в Kaggle отключен Internet, добавьте модель отдельным Kaggle Dataset и задайте `LOCAL_MODEL_PATH`, например `/kaggle/input/rubert-base-cased`.


In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

MODEL_NAME = "DeepPavlov/rubert-base-cased"
LOCAL_MODEL_PATH = None  # example: "/kaggle/input/rubert-base-cased"
MODEL_NAME_OR_PATH = LOCAL_MODEL_PATH or MODEL_NAME

MAX_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME_OR_PATH, use_fast=True)
print("Tokenizer loaded:", MODEL_NAME_OR_PATH)


## 9. Диагностика длины текстов

BERT принимает максимум 512 токенов. Мы оставляем `MAX_LENGTH = 512`, чтобы не терять важные фрагменты длинных Telegram-постов. Таблица ниже показывает, какая доля текстов будет обрезана.


In [ ]:
def estimate_token_lengths(texts: pd.Series, sample_size: int = 3000) -> pd.Series:
    sample = texts.sample(min(sample_size, len(texts)), random_state=SEED).tolist()
    encoded = tokenizer(sample, add_special_tokens=True, truncation=False, padding=False)
    return pd.Series([len(ids) for ids in encoded["input_ids"]], name="token_len")


lengths = estimate_token_lengths(train_df[TEXT_COL])
length_summary = lengths.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).to_frame().T
length_summary["share_over_max_length"] = (lengths > MAX_LENGTH).mean()
display(length_summary)


## 10. Torch Dataset

Используем обычный `torch.utils.data.Dataset`, чтобы не зависеть от `datasets`. Токенизация выполняется заранее без padding, а `DataCollatorWithPadding` динамически дополняет батчи до нужной длины. Это экономит память на Kaggle GPU.


In [ ]:
class TelegramRelevanceDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, tokenizer: AutoTokenizer, max_length: int = MAX_LENGTH):
        self.frame = frame.reset_index(drop=True)
        self.encodings = tokenizer(
            self.frame[TEXT_COL].tolist(),
            truncation=True,
            max_length=max_length,
            padding=False,
        )
        self.labels = self.frame[TARGET_COL].astype("float32").map(lambda x: [float(x)]).tolist()

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, idx: int) -> dict:
        item = {key: values[idx] for key, values in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item


train_dataset = TelegramRelevanceDataset(train_df, tokenizer, MAX_LENGTH)
val_dataset = TelegramRelevanceDataset(val_df, tokenizer, MAX_LENGTH)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)

sample = train_dataset[0]
print("Dataset sizes:", len(train_dataset), len(val_dataset))
print("Sample keys:", sample.keys())
print("Sample label:", sample["labels"])


## 11. Метрики

Во время обучения лучшую модель выбираем по `pr_auc` - это устойчивее при сильном дисбалансе классов. F1/precision/recall считаются при пороге `0.5`, а после обучения ниже отдельно подбирается лучший порог на validation.


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    fbeta_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

EVAL_THRESHOLD = 0.5


def sigmoid_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    return 1.0 / (1.0 + np.exp(-x))


def binary_metrics_from_probs(labels: np.ndarray, probs: np.ndarray, threshold: float = EVAL_THRESHOLD) -> dict:
    labels = np.asarray(labels).reshape(-1).astype(int)
    probs = np.asarray(probs).reshape(-1)
    preds = (probs >= threshold).astype(int)

    metrics = {
        "threshold": float(threshold),
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "f1": f1_score(labels, preds, zero_division=0),
        "f2": fbeta_score(labels, preds, beta=2, zero_division=0),
        "pred_positive_share": float(preds.mean()),
        "true_positive_share": float(labels.mean()),
    }

    try:
        metrics["pr_auc"] = average_precision_score(labels, probs)
    except Exception:
        metrics["pr_auc"] = 0.0

    try:
        metrics["roc_auc"] = roc_auc_score(labels, probs)
    except Exception:
        metrics["roc_auc"] = 0.0

    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    metrics.update({"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)})
    return metrics


def compute_metrics(eval_pred) -> dict:
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    probs = sigmoid_np(logits).reshape(-1)
    return binary_metrics_from_probs(labels, probs, threshold=EVAL_THRESHOLD)


## 12. Модель и взвешенная loss-функция

Для бинарной задачи используем `num_labels=1` и `BCEWithLogitsLoss`. `pos_weight = n_negative / n_positive` компенсирует редкий позитивный класс.


In [ ]:
import torch.nn.functional as F

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_OR_PATH,
    num_labels=1,
)
model.config.problem_type = "multi_label_classification"

n_pos = int(train_df[TARGET_COL].sum())
n_neg = int(len(train_df) - n_pos)
if n_pos == 0:
    raise ValueError("В train нет позитивных примеров.")

POS_WEIGHT_VALUE = n_neg / n_pos
FOCAL_GAMMA = None  # example: 2.0 для focal loss; None = weighted BCE

print("Train positives:", n_pos)
print("Train negatives:", n_neg)
print("pos_weight:", round(POS_WEIGHT_VALUE, 4))


class WeightedBCETrainer(Trainer):
    def __init__(self, *args, pos_weight_value: float = 1.0, focal_gamma: float | None = None, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight_value = float(pos_weight_value)
        self.focal_gamma = focal_gamma

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels").float()
        outputs = model(**inputs)
        logits = outputs.logits

        if labels.ndim == 1:
            labels = labels.view(-1, 1)
        labels = labels.to(logits.device)

        pos_weight = torch.tensor([self.pos_weight_value], dtype=logits.dtype, device=logits.device)

        if self.focal_gamma is None:
            loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
            loss = loss_fn(logits, labels)
        else:
            bce = F.binary_cross_entropy_with_logits(logits, labels, reduction="none", pos_weight=pos_weight)
            probs = torch.sigmoid(logits)
            p_t = probs * labels + (1 - probs) * (1 - labels)
            loss = ((1 - p_t) ** self.focal_gamma * bce).mean()

        return (loss, outputs) if return_outputs else loss


## 13. TrainingArguments

Настройки рассчитаны на Kaggle GPU. При OOM уменьшите `per_device_train_batch_size` до `2` или `MAX_LENGTH` до `384`.


In [ ]:
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
MODEL_OUTPUT_DIR = WORK_DIR / "rubert_telegram_relevance"
LOG_DIR = WORK_DIR / "logs_telegram_relevance"

training_kwargs = {
    "output_dir": str(MODEL_OUTPUT_DIR),
    "save_strategy": "epoch",
    "learning_rate": 2e-5,
    "per_device_train_batch_size": 4,
    "per_device_eval_batch_size": 8,
    "gradient_accumulation_steps": 4,
    "num_train_epochs": 5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.06,
    "lr_scheduler_type": "cosine",
    "max_grad_norm": 1.0,
    "load_best_model_at_end": True,
    "metric_for_best_model": "pr_auc",
    "greater_is_better": True,
    "logging_dir": str(LOG_DIR),
    "logging_steps": 50,
    "save_total_limit": 2,
    "report_to": "none",
    "seed": SEED,
    "data_seed": SEED,
    "remove_unused_columns": False,
    "fp16": torch.cuda.is_available(),
    "dataloader_num_workers": 2,
}

training_signature = inspect.signature(TrainingArguments.__init__).parameters
if "eval_strategy" in training_signature:
    training_kwargs["eval_strategy"] = "epoch"
else:
    training_kwargs["evaluation_strategy"] = "epoch"

if "optim" in training_signature:
    training_kwargs["optim"] = "adamw_torch"

unsupported_args = sorted(key for key in training_kwargs if key not in training_signature)
if unsupported_args:
    print("TrainingArguments: unsupported args skipped:", unsupported_args)
    training_kwargs = {key: value for key, value in training_kwargs.items() if key in training_signature}

training_args = TrainingArguments(**training_kwargs)
print(training_args)


## 14. Trainer


In [ ]:
trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": train_dataset,
    "eval_dataset": val_dataset,
    "data_collator": data_collator,
    "compute_metrics": compute_metrics,
    "callbacks": [EarlyStoppingCallback(early_stopping_patience=2)],
    "pos_weight_value": POS_WEIGHT_VALUE,
    "focal_gamma": FOCAL_GAMMA,
}

trainer_signature = inspect.signature(Trainer.__init__).parameters
if "processing_class" in trainer_signature:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = WeightedBCETrainer(**trainer_kwargs)
print("Trainer готов")


## 15. Обучение

Запускайте эту ячейку уже на Kaggle с GPU. Лучшая модель выбирается по validation `pr_auc`.


In [ ]:
train_result = trainer.train()

BEST_MODEL_DIR = WORK_DIR / "rubert_telegram_relevance_best"
trainer.save_model(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)

with open(WORK_DIR / "train_metrics.json", "w", encoding="utf-8") as f:
    json.dump(train_result.metrics, f, ensure_ascii=False, indent=2)

print("Best model saved to:", BEST_MODEL_DIR)
print("Train metrics:", train_result.metrics)


## 16. Подбор порога на validation

У взвешенной loss вероятности часто требуют отдельного порога. Ниже выбираем порог, который максимизирует F1 на validation. При необходимости можно заменить критерий на F2, если важнее recall.


In [ ]:
def trainer_predict_proba(dataset: Dataset) -> tuple[np.ndarray, np.ndarray]:
    pred = trainer.predict(dataset)
    logits = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
    probs = sigmoid_np(logits).reshape(-1)
    labels = pred.label_ids.reshape(-1).astype(int)
    return probs, labels


def tune_threshold(labels: np.ndarray, probs: np.ndarray, metric: str = "f1") -> pd.DataFrame:
    rows = []
    for threshold in np.linspace(0.05, 0.95, 181):
        rows.append(binary_metrics_from_probs(labels, probs, threshold=float(threshold)))
    table = pd.DataFrame(rows)
    return table.sort_values(metric, ascending=False).reset_index(drop=True)


val_probs, val_labels = trainer_predict_proba(val_dataset)
threshold_table = tune_threshold(val_labels, val_probs, metric="f1")
BEST_THRESHOLD = float(threshold_table.loc[0, "threshold"])

print("Best threshold:", BEST_THRESHOLD)
display(threshold_table.head(15))

threshold_table.to_csv(WORK_DIR / "validation_threshold_table.csv", index=False, encoding="utf-8-sig")
with open(WORK_DIR / "best_threshold.json", "w", encoding="utf-8") as f:
    json.dump({"best_threshold": BEST_THRESHOLD, "selection_metric": "f1"}, f, ensure_ascii=False, indent=2)


## 17. Финальная оценка на test

Считаем две оценки:

- `test_clean`: только `annotated=True`, наиболее честная оценка по чистым меткам.
- `test_full`: весь test, включая `annotated=False` негативы; полезно для понимания поведения на полном потоке, но часть нулей может быть шумной.


In [ ]:
def build_dataset(frame: pd.DataFrame) -> TelegramRelevanceDataset:
    return TelegramRelevanceDataset(frame, tokenizer, MAX_LENGTH)


def add_predictions(frame: pd.DataFrame, probs: np.ndarray, threshold: float) -> pd.DataFrame:
    out = frame.copy().reset_index(drop=True)
    out["prob_relevant"] = probs
    out["pred_relevant"] = (out["prob_relevant"] >= threshold).astype("int8")
    out["true_relevant"] = out[TARGET_COL].astype("int8")

    conditions = [
        (out["true_relevant"] == 1) & (out["pred_relevant"] == 1),
        (out["true_relevant"] == 0) & (out["pred_relevant"] == 0),
        (out["true_relevant"] == 0) & (out["pred_relevant"] == 1),
        (out["true_relevant"] == 1) & (out["pred_relevant"] == 0),
    ]
    out["error_type"] = np.select(conditions, ["TP", "TN", "FP", "FN"], default="?")
    return out


def evaluate_frame(frame: pd.DataFrame, name: str, threshold: float = BEST_THRESHOLD) -> tuple[dict, pd.DataFrame]:
    if len(frame) == 0:
        print(f"{name}: empty")
        return {}, frame.copy()

    dataset = build_dataset(frame)
    probs, labels = trainer_predict_proba(dataset)
    metrics = binary_metrics_from_probs(labels, probs, threshold=threshold)
    metrics["rows"] = int(len(frame))
    metrics["split"] = name

    pred_df = add_predictions(frame, probs, threshold)

    report = classification_report(
        labels,
        pred_df["pred_relevant"].to_numpy(),
        labels=[0, 1],
        target_names=["not_relevant", "relevant"],
        zero_division=0,
        output_dict=True,
    )
    pd.DataFrame(report).transpose().to_csv(
        WORK_DIR / f"{name}_classification_report.csv",
        encoding="utf-8-sig",
    )

    pred_df.to_csv(WORK_DIR / f"{name}_predictions.csv", index=False, encoding="utf-8-sig")
    return metrics, pred_df


test_clean_df = test_raw[test_raw[ANNOTATED_COL]].copy().reset_index(drop=True)
test_full_df = test_raw.copy().reset_index(drop=True)

val_metrics_best, val_predictions = evaluate_frame(val_df, "validation", BEST_THRESHOLD)
test_clean_metrics, test_clean_predictions = evaluate_frame(test_clean_df, "test_clean", BEST_THRESHOLD)
test_full_metrics, test_full_predictions = evaluate_frame(test_full_df, "test_full", BEST_THRESHOLD)

metrics_summary = {
    "training_mode": TRAINING_MODE,
    "model_name_or_path": str(MODEL_NAME_OR_PATH),
    "max_length": MAX_LENGTH,
    "pos_weight": POS_WEIGHT_VALUE,
    "best_threshold": BEST_THRESHOLD,
    "validation": val_metrics_best,
    "test_clean": test_clean_metrics,
    "test_full": test_full_metrics,
}

with open(WORK_DIR / "metrics_summary.json", "w", encoding="utf-8") as f:
    json.dump(metrics_summary, f, ensure_ascii=False, indent=2)

metrics_df = pd.DataFrame([val_metrics_best, test_clean_metrics, test_full_metrics])
display(metrics_df)
print("Metrics and predictions saved to:", WORK_DIR)


## 18. Анализ ошибок

False Negative - релевантные посты, которые модель пропустила. False Positive - нерелевантные по метке посты, которые модель подняла как релевантные. Для `annotated=False` часть FP может быть не ошибкой модели, а шумом в нулях.


In [ ]:
def show_error_examples(predictions: pd.DataFrame, error_type: str, n: int = 10) -> pd.DataFrame:
    cols = [col for col in META_COLS + TOPIC_COLS if col in predictions.columns]
    cols += ["true_relevant", "pred_relevant", "prob_relevant", "error_type", TEXT_COL]
    sample = predictions[predictions["error_type"] == error_type].copy()
    if error_type == "FN":
        sample = sample.sort_values("prob_relevant", ascending=True)
    elif error_type == "FP":
        sample = sample.sort_values("prob_relevant", ascending=False)
    return sample[cols].head(n)


print("False Negatives on test_clean:")
display(show_error_examples(test_clean_predictions, "FN", n=10))

print("False Positives on test_clean:")
display(show_error_examples(test_clean_predictions, "FP", n=10))

print("False Positives on test_full with annotated=False, проверка возможного шума в нулях:")
fp_unannotated = test_full_predictions[
    (test_full_predictions["error_type"] == "FP") &
    (test_full_predictions[ANNOTATED_COL] == False)
].copy()
display(show_error_examples(fp_unannotated, "FP", n=10))


## 19. Inference-функция для новых Telegram-постов

После обучения можно передать строку или список строк и получить вероятность релевантности.


In [ ]:
def predict_relevance(texts: str | list[str], threshold: float = BEST_THRESHOLD, batch_size: int = 16) -> pd.DataFrame:
    if isinstance(texts, str):
        texts = [texts]

    normalized = [normalize_telegram_text(text) for text in texts]
    device = trainer.model.device
    trainer.model.eval()

    all_probs = []
    for start in range(0, len(normalized), batch_size):
        batch_texts = normalized[start:start + batch_size]
        batch = tokenizer(
            batch_texts,
            truncation=True,
            max_length=MAX_LENGTH,
            padding=True,
            return_tensors="pt",
        )
        batch = {key: value.to(device) for key, value in batch.items()}
        with torch.no_grad():
            logits = trainer.model(**batch).logits.detach().cpu().numpy()
        all_probs.extend(sigmoid_np(logits).reshape(-1).tolist())

    result = pd.DataFrame({
        "text": normalized,
        "prob_relevant": all_probs,
    })
    result["pred_relevant"] = (result["prob_relevant"] >= threshold).astype("int8")
    return result


# Пример после обучения:
# predict_relevance([
#     "ЦБ снова объяснил, почему высокая ключевая ставка сохранится надолго.",
#     "Сегодня в городе перекрыли движение из-за ремонта дороги.",
# ])


## 20. Что сохранить после Kaggle-запуска

Основные артефакты будут лежать в `/kaggle/working`:

- `rubert_telegram_relevance_best/` - лучшая модель и tokenizer;
- `metrics_summary.json` - метрики validation, test_clean и test_full;
- `best_threshold.json` - выбранный порог;
- `test_clean_predictions.csv` и `test_full_predictions.csv` - вероятности, предсказания и типы ошибок;
- `validation_threshold_table.csv` - таблица порогов.
